In [1]:
# Клонування репозиторію курсу
!git clone https://github.com/dmytroslav/NLP_Course.git
%cd NLP_Course

# Встановлення необхідних бібліотек та української мовної моделі для spaCy
!pip install spacy pandas scikit-learn
!python -m spacy download uk_core_news_sm

Cloning into 'NLP_Course'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 334 (delta 18), reused 43 (delta 12), pack-reused 283 (from 1)
Receiving objects: 100% (334/334), 2.18 MiB | 17.89 MiB/s, done.
Resolving deltas: 100% (145/145), done.
/content/NLP_Course
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 98.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('uk_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart r

In [2]:
import os
import re
import pandas as pd
import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Завантаження української моделі spaCy
nlp = spacy.load("uk_core_news_sm", disable=["ner", "parser"])

print("Всі бібліотеки успішно імпортовано.")

Всі бібліотеки успішно імпортовано.


In [3]:
def clean_text(text):
    """
    Функція очищення тексту на основі напрацювань ЛР2.
    Видаляє зайві пробіли, нормалізує пунктуацію та специфічні символи.
    """
    if not isinstance(text, str):
        return ""

    # Нормалізація пробілів та переносів
    text = re.sub(r'\s+', ' ', text)
    # Нормалізація апострофів та лапок
    text = re.sub(r"['`’ʼ]", "'", text)
    # Видалення залишків html-тегів або специфічного шуму, якщо є
    text = text.strip()

    return text

def lemmatize_text(text):
    """
    Функція лематизації тексту за допомогою spaCy на основі ЛР3.
    """
    if not text:
        return ""
    doc = nlp(text)
    # Збираємо леми токенів, ігноруючи чисту пунктуацію та пробіли
    lemmas = [token.lemma_ for token in doc if not token.is_punct and not token.is_space]
    return " ".join(lemmas)

def preprocess_pipeline(text):
    """
    Повний конвеєр обробки сирого вхідного тексту
    """
    cleaned = clean_text(text)
    lemmatized = lemmatize_text(cleaned)
    return lemmatized

print("Функції препроцесингу готові до роботи.")

Функції препроцесингу готові до роботи.


In [4]:
# Шлях очищеного датасету
data_path = 'data/processed_v2.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(r"Датасет успішно завантажено.")
    print(f"Загальна кількість рядків: {len(df)}")
else:
    # Фоллбек-генерація штучних даних, якщо файл відсутній під час першого тесту
    print("Основний датасет не знайдено, створюємо sample_data для перевірки pipeline...")
    sample_data = {
        'text': [
            "ЗСУ відбили атаку на Куп'янському напрямку, ворог відступив.",
            "Терміново! Всі в укриття, ворог летить! Шок, дивитись всім!",
            "У зв'язку з підвищенням ключової ставки росіянам доведеться сплачувати додатковий податок.",
            "Офіційне повідомлення Міністерства оборони про зміну графіків мобілізації.",
            "Шокуючі новини! Таємна зброя знищила весь штаб командування, сенсаційне відео за посиланням!"
        ],
        'label': [True, False, False, True, False]
    }
    df = pd.DataFrame(sample_data)
    # Застосовуємо препроцесинг для демонстраційного датасету
    df['text'] = df['text'].apply(preprocess_pipeline)

# Перевірка наявності необхідних колонок (використовуємо оригінальний або лематизований текст відповідно до ЛР7)
# В ЛР7 LinearSVC на char n-grams показав найкращий результат саме на чистому тексті.
X = df['text'].astype(str)
y = df['label'].astype(int)

# Розбиття на train та test (стратифіковане, фіксований random_state для відтворюваності з ЛР5)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Розмір навчальної вибірки: {len(X_train)}")
print(f"Розмір тестової вибірки: {len(X_test)}")

Датасет успішно завантажено.
Загальна кількість рядків: 10735
Розмір навчальної вибірки: 8588
Розмір тестової вибірки: 2147


In [5]:
print("Запуск векторизації за допомогою ТF-IDF (Character n-grams 3-5)...")

# Налаштування векторизатора відповідно до параметрів ЛР7 (найкраща модель)
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=50000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Навчання фінального класифікатора LinearSVC...")
# Класифікатор з балансуванням ваг класів, як у ЛР7
final_model = LinearSVC(class_weight='balanced', C=1.0, random_state=42, dual=False)
final_model.fit(X_train_vec, y_train)

# Оцінка якості на тестовій вибірці
y_pred = final_model.predict(X_test_vec)

print("\n=== РЕЗУЛЬТАТИ ОЦІНКИ МОДЕЛІ (TEST SET) ===")
print(classification_report(y_test, y_pred, target_names=['Fake/Propaganda (0)', 'True News (1)']))
print(f"Загальна точність (Accuracy): {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro-F1 Score: {f1_score(y_test, y_pred, average='macro'):.4f}")

Запуск векторизації за допомогою ТF-IDF (Character n-grams 3-5)...
Навчання фінального класифікатора LinearSVC...

=== РЕЗУЛЬТАТИ ОЦІНКИ МОДЕЛІ (TEST SET) ===
                     precision    recall  f1-score   support

Fake/Propaganda (0)       0.84      0.82      0.83       500
      True News (1)       0.94      0.95      0.95      1647

           accuracy                           0.92      2147
          macro avg       0.89      0.88      0.89      2147
       weighted avg       0.92      0.92      0.92      2147

Загальна точність (Accuracy): 0.9208
Macro-F1 Score: 0.8881


In [14]:
def analyze_news_full(raw_text):
    """
    Комплексний аналіз новини: Класифікація (ЛР7) + Information Extraction (ЛР10)
    """
    if not raw_text or not raw_text.strip():
        return "Введіть коректний текст для аналізу."

    # 1. ПРЕПРОЦЕСИНГ ДЛЯ КЛАСИФІКАТОРА
    clean_and_lemma = preprocess_pipeline(raw_text)
    vec_text = vectorizer.transform([clean_and_lemma])

    # 2. КЛАСИФІКАЦІЯ ТА ВПЕВНЕНІСТЬ (CONFIDENCE)
    prediction = final_model.predict(vec_text)[0]
    # Оскільки LinearSVC не дає predict_proba напряму, використовуємо decision_function
    decision_score = final_model.decision_function(vec_text)[0]
    verdict = "TRUE (Правдива новина)" if prediction == 1 else "FAKE (Пропаганда / Маніпуляція)"

    # 3. ВИТЯГНЕННЯ СУТНОСТЕЙ (NER з ЛР10)
    # Використовуємо наш завантажений spaCy nlp
    doc = nlp(raw_text)
    entities = {}
    for ent in doc.ents:
        if ent.label_ not in entities:
            entities[ent.label_] = []
        entities[ent.label_].append(ent.text)

    # ДРУК РЕЗУЛЬТАТІВ У ГАРНОМУ ФОРМАТІ
    print("=" * 60)
    print("📰 АНАЛІЗ НОВИНИ")
    print("=" * 60)
    print(f"Текст: {raw_text}\n")
    print(f"Вердикт:      {verdict}")
    print(f"Сила сигналу: {abs(decision_score):.4f} (чим вище, тим впевненіша модель)")
    print("-" * 60)
    print("🔍 ЗНАЙДЕНІ СУТНОСТІ (Information Extraction):")

    if not entities:
        print("  Сутностей не знайдено.")
    else:
        for label, texts in entities.items():
            # Переклад стандартних тегів spaCy для гарного вигляду
            label_ua = {"ORG": "Організації", "LOC": "Локації", "PER": "Особи", "MONEY": "Гроші"}.get(label, label)
            print(f"  {label_ua}: {', '.join(set(texts))}")
    print("=" * 60)

# Тестуємо
analyze_news_full("Зеленський віддав наказ Міноборони виділити 5 мільйонів гривень на оборону Києва")

📰 АНАЛІЗ НОВИНИ
Текст: Зеленський віддав наказ Міноборони виділити 5 мільйонів гривень на оборону Києва

Вердикт:      TRUE (Правдива новина)
Сила сигналу: 1.1548 (чим вище, тим впевненіша модель)
------------------------------------------------------------
🔍 ЗНАЙДЕНІ СУТНОСТІ (Information Extraction):
  Сутностей не знайдено.


In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Створення поля для вводу тексту
text_input = widgets.Textarea(
    value='',
    placeholder='Вставте текст новини сюди...',
    description='Новина:',
    layout=widgets.Layout(width='90%', height='200px')
)

# Створення кнопки
button = widgets.Button(
    description='Перевірити',
    button_style='info',
    tooltip='Натисніть для аналізу',
    icon='search'
)

# Поле для виводу результату
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        # Викликаємо нашу функцію з попередньої комірки
        predict_news(text_input.value)

button.on_click(on_button_clicked)

# Відображення інтерфейсу
display(text_input, button, output)

Вхідний текст: Зеленський віддав всі окуповані території заради миру, а сам виїжджає закордон і більше не повернеться
Очищений/лематизований текст: зеленський віддати ввесь окупований територія заради мир а сам виїжджати закордон і більше не повернутися
Вердикт системи: TRUE (Правдива новина / Офіційне джерело)
--------------------------------------------------


In [19]:
!pip install groq ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 55.3 MB/s eta 0:00:00


In [20]:
import os
import getpass
from groq import Groq

print("="*60)
print("🔒 TIER 2: INITIALIZING GROQ CLIENT")
print("="*60)

# Безпечний запит ключа
api_key = getpass.getpass("Введіть ваш Groq API Key (або натисніть Enter, якщо вже налаштовано): ")

if api_key.strip():
    os.environ["GROQ_API_KEY"] = api_key
    print("✅ Ключ збережено в системних змінних.")
elif "GROQ_API_KEY" in os.environ:
    print("✅ Використовується існуючий GROQ_API_KEY із середовища.")
else:
    print("⚠️ Ключ не знайдено. Tier 2 буде пропущено.")

# Ініціалізація клієнта, якщо ключ доступний
client = None
if os.environ.get("GROQ_API_KEY"):
    try:
        client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
        AGENTS_ENABLED = True
        print("🚀 Клієнт Groq успішно ініціалізовано.")
    except Exception as e:
        print(f"❌ Помилка ініціалізації клієнта: {e}")
        AGENTS_ENABLED = False
else:
    AGENTS_ENABLED = False

🔒 TIER 2: INITIALIZING GROQ CLIENT
Введіть ваш Groq API Key (або натисніть Enter, якщо вже налаштовано): ··········
✅ Ключ збережено в системних змінних.
🚀 Клієнт Groq успішно ініціалізовано.


In [23]:
import json

# Визначення суворої схеми відповіді
JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "verdict": {"type": "string", "enum": ["True", "Fake", "Unknown"]},
        "confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
        "reasoning": {"type": "string"}
    },
    "required": ["verdict", "confidence", "reasoning"]
}

def call_llm_agent(text, feedback_prompt=None):
    """
    Прямий виклик моделі через Groq API з системним промптом контролю схеми.
    """
    system_instruction = (
        "You are an expert fact-checker analyzing Ukrainian news. "
        "You must evaluate the input and return text strictly matching this JSON schema:\n"
        f"{json.dumps(JSON_SCHEMA, ensure_ascii=False)}\n"
        "Do not include any conversational intro or outro. Return ONLY valid JSON."
    )

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"Analyze this text: {text}"}
    ]

    # Якщо це повторна ітерація (fallback/repair loop) — додаємо контекст помилки
    if feedback_prompt:
        messages.append({"role": "assistant", "content": "Error detected in format."})
        messages.append({"role": "user", "content": f"Fix the error: {feedback_prompt}"})

    try:
        completion = client.chat.completions.create(
            model="llama-3.1-8b-instant", # <--- ОСЬ ТУТ МИ ОНОВИЛИ МОДЕЛЬ
            messages=messages,
            temperature=0.1
        )
        return completion.choices[0].message.content
    except Exception as e:
        print(f"[Error] API call failed: {e}")
        return None

def validate_and_repair_flow(raw_text, max_attempts=2):
    """
    Stateful Flow: Маршрутизація -> Генерація -> Валідація схеми -> Ремонт (ЛР14)
    """
    attempt = 0
    feedback = None

    print(f"\n[Flow] Запуск аналізу для тексту: \"{raw_text[:40]}...\"")

    while attempt < max_attempts:
        attempt += 1
        print(f"[Flow] Спроба {attempt}: Запит до LLM Agent...")

        response_text = call_llm_agent(raw_text, feedback_prompt=feedback)
        if not response_text:
            feedback = "Empty output or connection error."
            continue

        # Спроба парсингу та валідації структури
        try:
            # Очищення від можливих маркдаун-обгорток ```json ... ```
            cleaned_response = re.sub(r'```json\s*|```', '', response_text).strip()
            parsed_json = json.loads(cleaned_response)

            # Валідація обов'язкових полів (Schema Validation)
            missing_fields = [field for field in JSON_SCHEMA["required"] if field not in parsed_json]
            if missing_fields:
                raise ValueError(f"missing_required_field: {missing_fields}")

            if parsed_json["verdict"] not in JSON_SCHEMA["properties"]["verdict"]["enum"]:
                raise ValueError(f"schema_validation_failure: invalid enum value for verdict")

            print("✅ [Validation] Успішно! Структура відповідає JSON контракту.")
            return parsed_json

        except (json.JSONDecodeError, ValueError) as err:
            print(f"⚠️ [Validation Failed] Виявлено помилку: {err}")
            # Формування чіткого фідбеку для наступного кроку repair loop
            feedback = f"Your previous output caused an error: {str(err)}. Output exactly valid JSON according to schema."

    print("❌ [Flow] Всі спроби вичерпано. Перехід у статус manual_review.")
    return {"verdict": "Unknown", "confidence": 0.0, "reasoning": "Failed to generate valid schema after repair loop."}

def check_news_two_tier(raw_text):
    """
    Головний інтерфейс Two-Tier архітектури проєкту
    """
    # Рівень 1: Швидкий класифікатор
    print("\n" + "="*40)
    print("▶ TIER 1: FAST ML FILTER (LinearSVC)")
    print("="*40)
    analyze_news_full(raw_text)

    # Рівень 2: Глибокий Агентний Флоу (якщо підключено API)
    if AGENTS_ENABLED:
        print("\n" + "="*40)
        print("▶ TIER 2: STATEFUL AGENTIC FLOW (LLM)")
        print("="*40)
        agent_output = validate_and_repair_flow(raw_text)

        print("\n📊 Фінальний вердикт Агента:")
        print(f"  Verdict:    {agent_output.get('verdict')}")
        print(f"  Confidence: {agent_output.get('confidence')}")
        print(f"  Reasoning:  {agent_output.get('reasoning')}")
    else:
        print("\nℹ️ Tier 2 вимкнено (відсутній API-ключ).")

# Тестування на семантично складному фейку, де класичний ML помилявся
if AGENTS_ENABLED:
    check_news_two_tier("Зеленський віддав всі окуповані території заради миру, а сам виїжджає закордон і більше не повернеться")


▶ TIER 1: FAST ML FILTER (LinearSVC)
📰 АНАЛІЗ НОВИНИ
Текст: Зеленський віддав всі окуповані території заради миру, а сам виїжджає закордон і більше не повернеться

Вердикт:      TRUE (Правдива новина)
Сила сигналу: 1.4983 (чим вище, тим впевненіша модель)
------------------------------------------------------------
🔍 ЗНАЙДЕНІ СУТНОСТІ (Information Extraction):
  Сутностей не знайдено.

▶ TIER 2: STATEFUL AGENTIC FLOW (LLM)

[Flow] Запуск аналізу для тексту: "Зеленський віддав всі окуповані територі..."
[Flow] Спроба 1: Запит до LLM Agent...
⚠️ [Validation Failed] Виявлено помилку: Extra data: line 1 column 254 (char 253)
[Flow] Спроба 2: Запит до LLM Agent...
✅ [Validation] Успішно! Структура відповідає JSON контракту.

📊 Фінальний вердикт Агента:
  Verdict:    Fake
  Confidence: 0.8
  Reasoning:  Україна продовжує контролювати деякі території, захоплені російськими військами, зокрема Керченську стрілку та частину Херсонської області. Також немає жодних офіційних повідомлень про те, щ

In [24]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Створення поля для введення тексту новини користувачем
news_textarea = widgets.Textarea(
    value='',
    placeholder='Вставте сюди текст новини для комплексної перевірки (наприклад: "Зеленський віддав всі окуповані території заради миру")...',
    description='Текст:',
    layout=widgets.Layout(width='85%', height='140px')
)

# 2. Створення стилізованої кнопки запуску дворівневого пайплайну
analyze_button = widgets.Button(
    description='🚀 Запустити Two-Tier Аналіз',
    button_style='success',  # Зелений колір кнопки
    tooltip='Натисніть, щоб прогнати текст через класичний ML та LLM Агента',
    icon='check-circle',
    layout=widgets.Layout(width='300px', height='40px')
)

# 3. Область для динамічного виведення результатів обробки
display_output = widgets.Output()

def handle_analysis_click(b):
    """
    Обробник події натискання кнопки. Зчитує текст, очищує екран
    та послідовно викликає Tier 1 та Tier 2 аналіз.
    """
    with display_output:
        clear_output()
        input_text = news_textarea.value.strip()

        if not input_text:
            print("❌ Помилка: Поле вводу порожнє! Будь ласка, скопіюйте сюди текст новини.")
            return

        # Виклик головної функції Two-Tier архітектури, яку ми зібрали раніше
        check_news_two_tier(input_text)

# Прив'язка функції обробки до кнопки
analyze_button.on_click(handle_analysis_click)

# Оформлення інтерфейсу в акуратний вертикальний блок (VBox)
interface_layout = widgets.VBox([
    widgets.HTML(value="<h3><b>🧠 Інтерактивна демо-панель перевірки новин</b></h3>"),
    widgets.HTML(value="<p>Система проведе експрес-аналіз (Tier 1) та запустить Stateful Agentic Flow (Tier 2), якщо введено API-ключ.</p>"),
    news_textarea,
    widgets.HTML(value="<br>"),
    analyze_button,
    widgets.HTML(value="<br>"),
    display_output
])

# Відображення готового віджета користувачу
display(interface_layout)